<a href="https://colab.research.google.com/github/Integenpiyush/financial-sentiment-bilstm-bert/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('/content/Sentences_AllAgree.txt',
    sep='@',
    header=None,
    names=['text', 'sentiment'],
    encoding='latin-1'
)

# 4. Strip any accidental whitespace from text and labels
df['text'] = df['text'].str.strip()
df['sentiment'] = df['sentiment'].str.strip()

# 5. Add our sequence validation column from EDA conclusions
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print(f"--- Data Successfully Loaded into 02_preprocessing.ipynb ---")
print(f"Total Records: {df.shape[0]}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass sanity check:\n{df['sentiment'].value_counts()}")

--- Data Successfully Loaded into 02_preprocessing.ipynb ---
Total Records: 2264
Columns: ['text', 'sentiment', 'word_count']

Class sanity check:
sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

# 1. Label Encoding
label_encoder = LabelEncoder()
df['label_idx'] = label_encoder.fit_transform(df['sentiment'])
# Mapping: negative -> 0, neutral -> 1, positive -> 2

# 2. Stratified Split (Crucial due to the 61.4% class imbalance)
X_train, X_val, y_train, y_val = train_test_split(
    df['text'].values,
    df['label_idx'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label_idx'].values
)

# 3. Target transformations for Keras loss functions
y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes=3)
y_val_onehot = tf.keras.utils.to_categorical(y_val, num_classes=3)

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))

print("Computed Class Weights for Training:")
for cls, weight in class_weight_dict.items():
    print(f"Class {label_encoder.inverse_transform([cls])[0]} (Index {cls}): {weight:.4f}")

Computed Class Weights for Training:
Class negative (Index 0): 2.4945
Class neutral (Index 1): 0.5424
Class positive (Index 2): 1.3238


In [12]:
from tensorflow.keras.layers import TextVectorization

MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 50

# Instantiate Vectorizer
bilstm_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQ_LEN,
    standardize='lower_and_strip_punctuation' # Good baseline for LSTM sequence matching
)

# Adapt the vectorizer to our training text corpus
bilstm_vectorizer.adapt(X_train)

# Example Transformation
sample_text = ["Operating profit rose to EUR 13.1 mn."]
print("Vectorized representation:", bilstm_vectorizer(sample_text).numpy())

Vectorized representation: [[  30   16   64    6    7 2526   12    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0]]


In [14]:
from transformers import AutoTokenizer
import numpy as np

# 1. Load pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# 2. Updated function returning numpy format (Keras reads this perfectly)
def tokenize_for_distilbert(texts):
    return tokenizer(
        list(texts),
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='np'  # Changed from 'tf' to 'np'
    )

# 3. Tokenize partitions
X_train_bert = tokenize_for_distilbert(X_train)
X_val_bert = tokenize_for_distilbert(X_val)

# 4. Print dictionary keys and underlying shape to verify
print("--- DistilBERT Tokenization Complete ---")
print("Tokenizer output keys:", list(X_train_bert.keys()))
print("Input IDs shape:", X_train_bert['input_ids'].shape)
print("Attention Mask shape:", X_train_bert['attention_mask'].shape)

--- DistilBERT Tokenization Complete ---
Tokenizer output keys: ['input_ids', 'token_type_ids', 'attention_mask']
Input IDs shape: (1811, 50)
Attention Mask shape: (1811, 50)


## Day 3 — Preprocessing Complete

### Splitting & Stratification
- **Train/Validation Split:** 80/20 stratified split applied to preserve class balance.
- **Training set:** 1,811 samples
- **Validation set:** 453 samples

### Imbalance Mitigation (Class Weights)
To address the severe 4.6x class imbalance, computed custom training weights to heavily penalize minority class errors:
- **Negative (Index 0):** 2.4945 *(Highest penalty)*
- **Neutral (Index 1):** 0.5424 *(Lowest penalty)*
- **Positive (Index 2):** 1.3238

### Dual-Track Pipeline Execution

#### Track A: Bi-LSTM Preprocessing
- Configured a Keras `TextVectorization` layer with a vocabulary limit of `10,000` tokens and fixed `max_sequence_length = 50`.
- Text standardized via lowercasing and punctuation stripping to facilitate clean sequential tracking.

#### Track B: DistilBERT Preprocessing
- Utilized the pre-trained `distilbert-base-uncased` tokenizer.
- Extracted `input_ids` and `attention_mask` matrices formatted as stable NumPy arrays to natively stream into our TensorFlow execution loops.
- Retained original casing bounds and punctuation markers to preserve full context for transformer self-attention calculations.

### Assets Exported
The `label_encoder.pkl` and `bilstm_vectorizer_config.pkl` have been serialized into the `../models/` directory, ensuring consistent inference behavior during local evaluation and final Flask app deployment.